Step 1: Load Customer Master Dataset

Upload the customer master dataset into Databricks and verify the schema.

In [0]:
print("Set up for Databricks ready")

Set up for Databricks ready


In [0]:
df = spark.table("workspace.default.customer_master_csv")

In [0]:
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [0]:
df.printSchema()

root
 |-- Row ID: long (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



Step 2: Data Exploration

In [0]:
df.count()

9994

In [0]:
df.columns

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

Checking null values in every column

In [0]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



Checking for duplicates

In [0]:
total_rows = df.count()
distinct_rows = df.distinct().count()

print("Total Rows:", total_rows)
print("Distinct Rows:", distinct_rows)
print("Duplicate Rows:", total_rows - distinct_rows)

Total Rows: 9994
Distinct Rows: 9994
Duplicate Rows: 0


In [0]:
# Replace spaces with underscores in all column names
df = df.toDF(*[c.replace(" ", "_") for c in df.columns])

# Verify new column names
df.printSchema()

root
 |-- Row_ID: long (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



Step 3: Create Delta Table

In [0]:
# Writing the DataFrame df to a Delta Lake table named customer_master, replacing any existing data if the table already exists.
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_master")

In [0]:
spark.sql("SELECT * FROM customer_master").show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|     State|Postal_Code|Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

Step 4: Incremental Loading

In [0]:
# Reading the customer_incremental table from the Spark catalog into a DataFrame named incremental_df for further processing.
incremental_df = spark.table("workspace.default.customer_incremental")

In [0]:
# Replace spaces with underscores in all column names
incremental_df = incremental_df.toDF(*[c.replace(" ", "_") for c in incremental_df.columns])

# Verify new column names
incremental_df.printSchema()

root
 |-- Row_ID: long (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



Step 5: SCD Type 2

In [0]:
# It reads the customer_master table and adds *SCD Type 2* columns: is_current (set to True), effective_date (today's date), and end_date (initialized as NULL).
from pyspark.sql.functions import current_date, lit

master_df = spark.table("customer_master")

master_df = master_df \
    .withColumn("is_current", lit(True)) \
    .withColumn("effective_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date"))

In [0]:
master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("customer_master_scd2")

In [0]:
from delta.tables import DeltaTable

scd2 = DeltaTable.forName(spark, "customer_master_scd2")

In [0]:
scd2.alias("target").merge(
    incremental_df.alias("source"),
    "target.Row_ID = source.Row_ID AND target.is_current = true"
).whenMatchedUpdate(
    condition="""
        target.City <> source.City OR
        target.State <> source.State OR
        target.Sales <> source.Sales OR
        target.Quantity <> source.Quantity
    """,
    set={
        "is_current": "false",
        "end_date": "current_date()"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
changed_or_new = incremental_df.alias("source").join(
    spark.table("customer_master_scd2")
         .filter("is_current = true")
         .alias("target"),
    "Row_ID",
    "left"
).filter("""
target.Row_ID IS NULL
OR target.City <> source.City
OR target.State <> source.State
OR target.Sales <> source.Sales
OR target.Quantity <> source.Quantity
""").select("source.*")

In [0]:
changed_or_new = changed_or_new \
    .withColumn("is_current", lit(True)) \
    .withColumn("effective_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date"))

In [0]:
changed_or_new.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("customer_master_scd2")

Step 6: SCD Type 1

SCD Type 1 overwrites existing records.
Historical information is not preserved.

In [0]:
master_df = spark.table("customer_master")

master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("customer_master_scd1")

In [0]:
incremental_df.createOrReplaceTempView("incremental_data")

In [0]:
%sql
MERGE INTO customer_master_scd1 AS target
USING incremental_data AS source
ON target.Row_ID = source.Row_ID

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
16,16,0,0


In [0]:
spark.table("customer_master_scd1").count()

9994

In [0]:
spark.table("customer_master_scd1").show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|     State|Postal_Code|Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [0]:
spark.sql("""
SELECT *
FROM customer_master_scd1
WHERE Row_ID IN
(
SELECT Row_ID
FROM incremental_data
)
""").show()

+------+--------------+----------+----------+----------------+-----------+---------------+---------+-------------+---------------+--------------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|       Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|         State|Postal_Code|Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+----------------+-----------+---------------+---------+-------------+---------------+--------------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|    Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|      Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   

Step 7: Validation

In [0]:
print("Master:", spark.table("customer_master").count())
print("SCD1:", spark.table("customer_master_scd1").count())
print("SCD2:", spark.table("customer_master_scd2").count())

Master: 9994
SCD1: 9994
SCD2: 9994


In [0]:
spark.table("customer_master_scd2") \
    .filter("is_current = false") \
    .show()

+------+--------------+----------+----------+--------------+-----------+--------------+-----------+-------------+---------------+--------+-----------+-------+---------------+---------------+------------+--------------------+------+--------+--------+--------+----------+--------------+----------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID| Customer_Name|    Segment|      Country|           City|   State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name| Sales|Quantity|Discount|  Profit|is_current|effective_date|  end_date|
+------+--------------+----------+----------+--------------+-----------+--------------+-----------+-------------+---------------+--------+-----------+-------+---------------+---------------+------------+--------------------+------+--------+--------+--------+----------+--------------+----------+
|     2|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|   Claire Gute|   Consumer|United States

In [0]:
spark.table("customer_master_scd2") \
.filter("is_current = true") \
.show()

+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+----------+--------------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|     Customer_Name|  Segment|      Country|           City|         State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|  Profit|is_current|effective_date|end_date|
+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+----------+--------------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|       Claire Gu

In [0]:
spark.table("customer_master_scd1").show(5)

spark.table("customer_master_scd2").show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|     State|Postal_Code|Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

Conclusion

• Loaded customer master data.

• Created Delta tables.

• Created incremental dataset.

• Implemented SCD Type 1 using MERGE.

• Implemented SCD Type 2 to maintain historical records.

• Validated inserted and updated records.